In [13]:
import os
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
# Set JAVA_HOME to the path of Java 17
# (This command finds the path dynamically using the mac system tool)
java17_path = os.popen("/usr/libexec/java_home -v 17").read().strip()

if java17_path:
    os.environ["JAVA_HOME"] = java17_path
    print(f"Successfully set JAVA_HOME to: {java17_path}")
else:
    print("Error: Java 17 not found! Please verify installation.")

Successfully set JAVA_HOME to: /Library/Java/JavaVirtualMachines/jdk-17.jdk/Contents/Home


In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark aggregation functions") \
    .getOrCreate()

In [9]:
listings = spark.read.csv("../data/raw/listings.csv.gz", 
    header=True,
    inferSchema=True,
    sep=",", 
    quote='"',
    escape='"', 
    multiLine=True,
    mode="PERMISSIVE" 
)
listings.printSchema()

root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_listings_count: int

In [10]:
reviews = spark.read.csv("../data/raw/reviews.csv.gz", 
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)
reviews.printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)



In [15]:
# 1. For each listing compute string category depending on its price, and add it as a new column.
# A category is defined in the following way:
#
# * price < 50 -> "Budget"
# * 50 <= price < 150 -> "Mid-range"
# * price >= 150 -> "Luxury"
# 
# Only include listings where the price is not null.
# Count the number of listings in each category

from pyspark.sql.functions import regexp_replace

listings = listings.filter(listings.price_numeric.isNotNull()).withColumn('price_numeric', regexp_replace('price', '[$,]', '').cast('float'))

# TODO: Implement a UDF
def price_category(price_data):
    if price_data < 50:
        return 'Budget'
    elif price_data >= 50 and price_data < 150:
        return 'Mid-range'
    else:
        return 'Luxury'
        
# TODO: Apply the UDF to create a new DataFrame
categorize_price_udf=udf(price_category,StringType())
listings_with_category=listings.withColumn('price_category',categorize_price_udf(listings.price_numeric)).groupBy('price_category') \
.count().show()

+--------------+-----+
|price_category|count|
+--------------+-----+
|     Mid-range|28333|
|        Budget| 6114|
|        Luxury|27516|
+--------------+-----+



In [20]:
# 2. In this task you will need to compute a santiment score per review, and then an average sentiment score per listing.
# A santiment score indicates how "positive" or "negative" a review is. The higher the score the more positive it is, and vice-versa.
#
# To compute a sentiment score per review compute the number of positive words in a review and subtract the number of negative
# words in the same review (the list of words is already provided)
#
# To complete this task, compute a DataFrame that contains the following fields:
# * name - the name of a listing
# * average_sentiment - average sentiment of reviews computed using the algorithm described above
from pyspark.sql.types import FloatType
from pyspark.sql.functions import avg

# Lists of positive and negative words
positive_words = {'good', 'great', 'excellent', 'amazing', 'fantastic', 'wonderful', 'pleasant', 'lovely', 'nice', 'enjoyed'}
negative_words = {'bad', 'terrible', 'awful', 'horrible', 'disappointing', 'poor', 'hate', 'unpleasant', 'dirty', 'noisy'}

# TODO: Implement the UDF
def sentiment_score(comment):
    if comment is None:
        return 0
    comment=comment.lower()
    postive_count=0
    negative_count=0
    for word in comment:
        if word in positive_words:
            postive_count+=1

    for word in comment:
        if word in negative_words:
            negative_count+=1     
    
    return postive_count - negative_count
    

sentiment_score_udf = udf(sentiment_score, FloatType())

reviews_with_sentiment = reviews \
  .withColumn(
    'sentiment_score',
    sentiment_score_udf(reviews.comments)
  )

# TODO: Create a final DataFrame
listings.join(reviews_with_sentiment,listings.id==reviews_with_sentiment.listing_id,'inner') \
.groupBy('listing_id','name').agg(avg('sentiment_score').alias('average_sentiment_score')) \
.orderBy('average_sentiment_score',ascending=False) \
.select('listing_id','name','average_sentiment_score').show()

+----------+--------------------+-----------------------+
|listing_id|                name|average_sentiment_score|
+----------+--------------------+-----------------------+
|    810314|BALCONY/OVERLOOKI...|                   NULL|
|   1038126|Crayford - Camden...|                   NULL|
|   1208955|Relaxing Georgian...|                   NULL|
|   2867009|Calm, light space...|                   NULL|
|   3342278|Spacious Studio i...|                   NULL|
|   3930288|1st Double Room, ...|                   NULL|
|   3666520|GARDEN ROOM IN KE...|                   NULL|
|   4743954|Large elegant dou...|                   NULL|
|   5223640|Bright spacious d...|                   NULL|
|   6151998|Warwick avenue -1...|                   NULL|
|   7409745|Lovely garden fla...|                   NULL|
|   7635698|Comfy room in mar...|                   NULL|
|   7730230|Designer Loft in ...|                   NULL|
|   8549167|Cosy small double...|                   NULL|
|   8527086|Si

In [36]:
# 3. Rewrite the following code from the previous exercise using SparkSQL:
#
# ```
# from pyspark.sql.functions import length, avg, count
# 
# reviews_with_comment_length = reviews.withColumn('comment_length', length('comments'))
# reviews_with_comment_length \
#   .join(listings, reviews_with_comment_length.listing_id == listings.id, 'inner') \
#   .groupBy('listing_id').agg(
#       avg(reviews_with_comment_length.comment_length).alias('average_comment_length'),
#       count(reviews_with_comment_length.id).alias('reviews_count')
#   ) \
#   .filter('reviews_count >= 5') \
#   .orderBy('average_comment_length', ascending=False) \
#   .show()
# ```
# This was a solution for the the task:
#
# "Get top five listings with the highest average review comment length. Only return listings with at least 5 reviews"

reviews.createOrReplaceTempView("reviews")
listings.createOrReplaceTempView("listings")

# Write the SQL query
sql_query = """
select listings.id,avg(length(reviews.comments)) average_comment_length
from listings join reviews on listings.id=reviews.listing_id
group by listings.id
having count(reviews.id) >= 5
order by average_comment_length
"""

spark \
  .sql(sql_query) \
  .explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [average_comment_length#1011 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(average_comment_length#1011 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=1416]
      +- Project [id#85L, average_comment_length#1011]
         +- Filter (count(id)#1014L >= 5)
            +- HashAggregate(keys=[id#85L], functions=[avg(length(comments#169)), count(id#165L)])
               +- Exchange hashpartitioning(id#85L, 200), ENSURE_REQUIREMENTS, [plan_id=1411]
                  +- HashAggregate(keys=[id#85L], functions=[partial_avg(length(comments#169)), partial_count(id#165L)])
                     +- Project [id#85L, id#165L, comments#169]
                        +- BroadcastHashJoin [id#85L], [listing_id#164L], Inner, BuildLeft, false
                           :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, true]),false), [plan_id=1406]
                           :  +- Project [id#85L]
          

In [ ]:
# 4. [Optional][Challenge]
# Calculate an average time passed from the first review for each host in the listings dataset. 
# To implmenet a custom aggregation function you would need to use "pandas_udf" function to write a custom aggregation function.
#
# Documentation about "pandas_udf": https://spark.apache.org/docs/3.4.2/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pandas_udf.html 
#
# To use "pandas_udf" you would need to install two additional dependencies in the virtual environment you use for PySpark:
# Run these commands:
# ```
# pip install pandas
# pip install pyarrow
# ```

from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import PandasUDFType
import pandas as pd

@pandas_udf(DoubleType(), functionType=PandasUDFType.GROUPED_AGG)
def average_days_since_first_review_udf(first_review_series) -> float:
    # TODO: Implement the UDF
    pass

listings \
  .filter(
    listings.first_review.isNotNull()
  ) \
  .groupBy('host_id') \
  .agg(
    average_days_since_first_review_udf(listings.first_review).alias('average_days_since_first_review_days')
  ) \
  .show()

In [32]:
myRange = spark.range(1000).toDF("number")
divisBy2 = myRange.where("number % 2 = 0")
divisBy2.show()

+------+
|number|
+------+
|     0|
|     2|
|     4|
|     6|
|     8|
|    10|
|    12|
|    14|
|    16|
|    18|
|    20|
|    22|
|    24|
|    26|
|    28|
|    30|
|    32|
|    34|
|    36|
|    38|
+------+
only showing top 20 rows


In [34]:
listings.take(2)

[Row(id=13913, listing_url='https://www.airbnb.com/rooms/13913', scrape_id=20250914034649, last_scraped=datetime.date(2025, 9, 16), source='city scrape', name='Holiday London DB Room Let-on going', description='My bright double bedroom with a large window has a relaxed feeling! It comfortably fits one or two and is centrally located just two blocks from Finsbury Park. Enjoy great restaurants in the area and easy access to easy transport tubes, trains and buses. Babies and children of all ages are welcome.', neighborhood_overview='Finsbury Park is a friendly melting pot community composed of Turkish, French, Spanish, Middle Eastern, Irish and English families. <br />We have a wonderful variety of international restaurants directly under us on Stroud Green Road. And there are many shops and large Tescos supermarket right next door. <br /><br />But you can also venture up to Crouch End and along Greens Lanes where there will endless choice of Turkish and Middle Eastern cuisines.s', pictur